# MELD multimodal emotion classifier
Use Python 3.10+ and install `requirements.txt` into the selected notebook kernel before running all cells (see README.md).
Training requires the MELD CSV files **and extracted video clips**. Configure `data_dir` and `video_dirs` below for your installation. The first run downloads pretrained models from Hugging Face.

In [ ]:
%pip install -r requirements.txt
# Restart the kernel if packages were changed, then continue with the next cell.

In [ ]:
import torch
from transformers.utils.import_utils import (
    get_torch_version,
    is_torch_greater_or_equal,
)

print(torch.__version__)
print(get_torch_version())
print(is_torch_greater_or_equal("2.6"))

2.6.0+cu124
2.6.0+cu124
True


In [2]:
import numpy as np
import torch
import torch.nn as nn


In [ ]:
import pandas as pd
import os

data_dir = "datasets/MELD"

train = pd.read_csv(os.path.join(data_dir, "train_sent_emo.csv"))
# print(train["Utterance"][0])
train


,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,also I was the point person on my company’s tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731"
1,2,You must’ve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442"
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389"
3,4,So let’s talk a little bit about your duties.,The Interviewer,neutral,neutral,0,3,8,21,"00:16:26,820","00:16:29,572"
4,5,My duties? All right.,Chandler,surprise,positive,0,4,8,21,"00:16:34,452","00:16:40,917"
...,...,...,...,...,...,...,...,...,...,...,...
9984,10474,You or me?,Chandler,neutral,neutral,1038,13,2,3,"00:00:48,173","00:00:50,799"
9985,10475,"I got it. Uh, Joey, women don't have Adam's ap...",Ross,neutral,neutral,1038,14,2,3,"00:00:51,009","00:00:53,594"
9986,10476,"You guys are messing with me, right?",Joey,surprise,positive,1038,15,2,3,"00:01:00,518","00:01:03,520"
9987,10477,Yeah.,All,neutral,neutral,1038,16,2,3,"00:01:05,398","00:01:07,274"


In [14]:
from torch.utils.data import Dataset, DataLoader
import os



class MELDDataset(Dataset):
    def __init__(self, data_path, mode_dir):
        df = pd.read_csv(data_path)
        self.data = []
        for i in range(len(df)):
            utterance = df["Utterance"][i]
            mp4_path = os.path.join(mode_dir, f"dia{df['Dialogue_ID'][i]}_utt{df['Utterance_ID'][i]}.mp4")

            emotion = df["Emotion"][i]
            to_append = {
                "text": utterance,
                "visual": mp4_path,
                "label": emotion
            }
            self.data.append(to_append)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, key):
        return self.data[key]

train_data_path = os.path.join(data_dir, "train_sent_emo.csv")
dev_data_path = os.path.join(data_dir, "dev_sent_emo.csv")
test_data_path = os.path.join(data_dir, "test_sent_emo.csv")

# Point these folders at the extracted MELD clips if their names differ.
video_dirs = {
    "train": os.path.join(data_dir, "train"),
    "dev": os.path.join(data_dir, "dev"),
    "test": os.path.join(data_dir, "test"),
}

train_dataset = MELDDataset(train_data_path, mode_dir=video_dirs["train"])
dev_dataset = MELDDataset(dev_data_path, mode_dir=video_dirs["dev"])
test_dataset = MELDDataset(test_data_path, mode_dir=video_dirs["test"])
# These two files are unusable in the distributed media: one is corrupt and
# one is absent. Keep the official CSV annotations unchanged and filter here.
known_unavailable_files = {
    "train": {"dia125_utt3.mp4"},
    "dev": {"dia110_utt7.mp4"},
}
for split, dataset in (
    ("train", train_dataset),
    ("dev", dev_dataset),
    ("test", test_dataset),
):
    excluded = known_unavailable_files.get(split, set())
    dataset.data = [
        sample
        for sample in dataset.data
        if os.path.basename(sample["visual"]) not in excluded
    ]


missing_videos = [
    sample["visual"]
    for dataset in (train_dataset, dev_dataset, test_dataset)
    for sample in dataset
    if not os.path.isfile(sample["visual"])
]
if missing_videos:
    raise FileNotFoundError(
        f"Missing {len(missing_videos)} MELD video clips (first: {missing_videos[0]}). "
        "Extract the MELD videos and update video_dirs above. The CSV files alone "
        "are not enough for multimodal training. See README.md."
    )

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
test_data_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import cv2
from transformers import (
    AutoModel,
    AutoTokenizer,
    VideoMAEImageProcessor,
    VideoMAEModel,
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
video_processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")

emotions_to_id = {
    "anger": 0,
    "disgust": 1,
    "fear": 2,
    "joy": 3,
    "neutral": 4,
    "sadness": 5,
    "surprise": 6,
}

cv2.setNumThreads(1)
video_pool = ThreadPoolExecutor(max_workers=4)


def read_video(path, num_frames):
    capture = cv2.VideoCapture(path)
    try:
        if not capture.isOpened():
            raise ValueError(f"Cannot open video: {path}")

        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            raise ValueError(f"Video has no readable frames: {path}")

        target_indices = np.linspace(
            0,
            frame_count - 1,
            num_frames,
        ).astype(int)

        frames = []
        target_position = 0
        for frame_index in range(frame_count):
            ok, frame = capture.read()
            if not ok:
                break

            while (
                target_position < num_frames
                and target_indices[target_position] == frame_index
            ):
                frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                target_position += 1

            if target_position == num_frames:
                break

        if len(frames) != num_frames:
            raise ValueError(
                f"Decoded {len(frames)} of {num_frames} frames from {path}"
            )
        return frames
    finally:
        capture.release()


class MultimodalModel(nn.Module):
    def __init__(self, text_model, visual_model):
        super().__init__()
        self.text_model = text_model
        self.visual_model = visual_model

        # Only train the fusion classifier.
        for param in self.text_model.parameters():
            param.requires_grad = False
        for param in self.visual_model.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(
                text_model.config.hidden_size + visual_model.config.hidden_size,
                512,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, len(emotions_to_id)),
        )

    def train(self, mode=True):
        super().train(mode)
        # Frozen encoders should remain deterministic; classifier dropout still trains.
        self.text_model.eval()
        self.visual_model.eval()
        return self

    def forward(self, text, visual):
        device = next(self.classifier.parameters()).device
        text_tokens = tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        num_frames = self.visual_model.config.num_frames
        frames = list(
            video_pool.map(
                lambda path: read_video(path, num_frames),
                visual,
            )
        )
        visual_tokens = video_processor(frames, return_tensors="pt").to(device)

        with torch.no_grad():
            text_hidden = self.text_model(**text_tokens).last_hidden_state
            mask = text_tokens["attention_mask"].unsqueeze(-1).to(text_hidden.dtype)
            text = (text_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            visual = self.visual_model(
                **visual_tokens
            ).last_hidden_state.mean(dim=1)

        combined = torch.cat([text, visual], dim=-1)
        return self.classifier(combined)


In [28]:
# Training loop
text_model = AutoModel.from_pretrained(
    "microsoft/deberta-v3-small"
)



visual_model = VideoMAEModel.from_pretrained(
    "MCG-NJU/videomae-base"
)

model = MultimodalModel(
    text_model=text_model,
    visual_model=visual_model
)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print("Device:", device)

num_epochs = 10
learning_rate = 5e-4

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=learning_rate)
# criterion = nn.CrossEntropyLoss()
train_label_ids = torch.tensor([emotions_to_id[sample["label"]] for sample in train_dataset])
class_counts = torch.bincount(train_label_ids, minlength = len(emotions_to_id))
balanced_weights = (
    len(train_label_ids)
    / (len(emotions_to_id) * class_counts)
)
class_weights = torch.sqrt(balanced_weights).to(device)


print({
    emotion: round(class_weights[label_id].item(), 3)
    for emotion, label_id in emotions_to_id.items()
})

criterion = nn.CrossEntropyLoss(weight=class_weights)


def batch_labels(batch):
    return torch.tensor([emotions_to_id[label] for label in batch["label"]], dtype=torch.long, device=device)


@torch.no_grad()
def accuracy(loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        # Convert dataset labels to numeric IDs.
        labels = batch_labels(batch)
        # Text and video are preprocessed inside the model.
        predictions = model(batch["text"], batch["visual"]).argmax(dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    return correct / total



for epoch in range(num_epochs):
    model.train()
    total_loss, total_examples = 0.0, 0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = model(batch["text"], batch["visual"])
        # Convert dataset labels to numeric IDs.
        labels = batch_labels(batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)

    print(f"Epoch {epoch + 1}/{num_epochs} | loss: {total_loss / total_examples:.4f} | "
          f"validation accuracy: {accuracy(dev_loader):.2%}")

print(f"Test accuracy: {accuracy(test_data_loader):.2%}")
# Only the classifier learns; reuse the same pretrained encoders when loading it.
torch.save({"classifier": model.classifier.state_dict(), "labels": emotions_to_id}, "meld_classifier_weighted.pt")


Device: cuda
{'anger': 1.134, 'disgust': 2.295, 'fear': 2.307, 'joy': 0.905, 'neutral': 0.55, 'sadness': 1.445, 'surprise': 1.088}
Epoch 1/10 | loss: 1.5523 | validation accuracy: 54.87%
Epoch 2/10 | loss: 1.3845 | validation accuracy: 58.30%
Epoch 3/10 | loss: 1.3265 | validation accuracy: 58.03%
Epoch 4/10 | loss: 1.2892 | validation accuracy: 58.66%
Epoch 5/10 | loss: 1.2516 | validation accuracy: 57.67%
Epoch 6/10 | loss: 1.2187 | validation accuracy: 54.96%
Epoch 7/10 | loss: 1.1864 | validation accuracy: 56.77%
Epoch 8/10 | loss: 1.1401 | validation accuracy: 59.66%
Epoch 9/10 | loss: 1.1159 | validation accuracy: 55.23%
Epoch 10/10 | loss: 1.0823 | validation accuracy: 58.48%
Test accuracy: 59.08%


In [30]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

checkpoint = torch.load("meld_classifier_weighted.pt", map_location=device, weights_only=True)
model = MultimodalModel(text_model=text_model, visual_model=visual_model).to(device)
model.classifier.load_state_dict(checkpoint["classifier"])
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for batch in test_data_loader:
        labels = batch_labels(batch)
        predictions = model(batch["text"], batch["visual"]).argmax(dim=-1)
        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())

label_ids = list(range(len(emotions_to_id)))
emotion_names = [
    emotion
    for emotion, label_id in sorted(emotions_to_id.items(), key=lambda item: item[1])
]

print(f"Accuracy: {accuracy_score(all_labels, all_predictions):.2%}")
print(f"Weighted F1: {f1_score(all_labels, all_predictions, average='weighted', zero_division=0):.2%}")
print(f"Macro F1: {f1_score(all_labels, all_predictions, average='macro', zero_division=0):.2%}")
print("\nPer-class metrics:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=label_ids,
    target_names=emotion_names,
    digits=4,
    zero_division=0,
))


Accuracy: 59.08%
Weighted F1: 59.25%
Macro F1: 41.40%

Per-class metrics:
              precision    recall  f1-score   support

       anger     0.4307    0.5043    0.4646       345
     disgust     0.2273    0.0735    0.1111        68
        fear     0.1321    0.2800    0.1795        50
         joy     0.5612    0.5249    0.5424       402
     neutral     0.7705    0.7166    0.7426      1256
     sadness     0.3548    0.2644    0.3030       208
    surprise     0.4828    0.6512    0.5545       281

    accuracy                         0.5908      2610
   macro avg     0.4228    0.4307    0.4140      2610
weighted avg     0.6029    0.5908    0.5925      2610

